In [2]:
import torch
import torch.nn as nn

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [24]:

class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model):
        super().__init__()

        # TODO 1: embedding для токенов
        self.token_embedding = nn.Embedding(vocab_size, d_model)

        # TODO 2: embedding для позиций
        self.position_embedding = nn.Embedding(max_seq_len, d_model)

    def forward(self, token_ids):
        # token_ids: (B, T)
        B, T = token_ids.shape

        # TODO 3: создать позиции от 0 до T - 1
        positions = torch.arange(0, T).to(device=device)

        # TODO 4: получить векторы токенов и позиций
        token_emb = self.token_embedding(token_ids)
        position_emb = self.position_embedding(positions)


        # TODO 5: сложить embeddings
        x = token_emb + position_emb

        return x

In [5]:
model = TokenAndPositionEmbedding(
    vocab_size=100,
    max_seq_len=20,
    d_model=16
)

In [6]:
token_ids = torch.tensor(
    [[1, 2, 3, 4, 5],
     [6, 7, 8, 9, 10]]
)

In [7]:
token_ids.shape

torch.Size([2, 5])

In [8]:
output = model(token_ids)

torch.Size([2, 5, 16]) torch.Size([5, 16])


In [9]:
print("Input shape:", token_ids.shape)
print("Output shape:", output.shape)
print("Number of parameters:", sum(p.numel() for p in model.parameters()))

assert output.shape == (2,5,16)
assert torch.isfinite(output).all

Input shape: torch.Size([2, 5])
Output shape: torch.Size([2, 5, 16])
Number of parameters: 1920


### Self-Attention

In [10]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)

    def forward(self, x):
        # x: (B, T, D)

        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # TODO 4: оценки сходства
        scores = Q @ K.transpose(-2, -1)

        # TODO 5: масштабирование и Softmax по последней оси
        weights = F.softmax(scores / math.sqrt(Q.shape[-1]), dim=-1)

        # TODO 6: взвешенная сумма Value
        output = weights @ V

        return output

In [11]:
torch.manual_seed(42)

attention = SelfAttention(d_model=16)
x = torch.randn(2, 5, 16)

output = attention(x)

print(output.shape)
assert output.shape == (2, 5, 16)
assert torch.isfinite(output).all()

torch.Size([2, 5, 16])


### Multi-Head Attention

In [12]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape
        H = self.num_heads
        Dh = self.head_dim

        # TODO 1: вычисли Q, K, V
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # TODO 2: раздели каждую проекцию на головы
        # (B, T, D) -> (B, H, T, Dh)
        Q = Q.view(B,T, H, Dh).transpose(1, 2)
        K = K.view(B,T, H, Dh).transpose(1, 2)  
        V = V.view(B,T, H, Dh).transpose(1, 2)

        # TODO 3: вычисли attention scores
        # (B, H, T, Dh) @ (B, H, Dh, T)
        # -> (B, H, T, T)
        scores = Q @ K.transpose(-2,-1)

        # TODO 4: масштабирование и Softmax
        weights = F.softmax(scores / math.sqrt(Dh), dim=-1)

        # TODO 5: умножь веса на V
        # Результат: (B, H, T, Dh)
        output = weights @ V

        # TODO 6: объедини головы обратно
        # (B, H, T, Dh) -> (B, T, D)
        output = output.transpose(1,2).reshape(B,T,D)

        output = self.out_proj(output)

        # TODO 7: выходная проекция
        return output

In [13]:
torch.manual_seed(42)

model = MultiHeadAttention(d_model=16, num_heads=4)
x = torch.randn(2, 5, 16)

output = model(x)

print(output.shape)
assert output.shape == (2, 5, 16)
assert torch.isfinite(output).all()

loss = output.square().mean()
loss.backward()

assert model.query.weight.grad is not None
assert model.key.weight.grad is not None
assert model.value.weight.grad is not None
assert model.out_proj.weight.grad is not None

print("Forward и backward прошли успешно!")

torch.Size([2, 5, 16])
Forward и backward прошли успешно!


### TransformerBlock

In [18]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * mlp_ratio),
            nn.GELU(),
            nn.Linear(d_model * mlp_ratio, d_model)
        )

    def forward(self, x):
        x = self.attention(self.ln1(x)) + x

        x = self.ffn(self.ln2(x)) + x 

        return x

In [19]:
torch.manual_seed(42)

block = TransformerBlock(d_model=16, num_heads=4)
x = torch.randn(2,5,16)

y = block(x)

print("Input: ", x.shape)
print("Output: ", y.shape)

assert x.shape == y.shape
assert torch.isfinite(y).all()

loss = y.square().mean()
loss.backward()

assert block.attention.query.weight.grad is not None
assert block.ffn[0].weight.grad is not None
assert block.ffn[2].weight.grad is not None

print("Forward и backward прошли успешно!")


Input:  torch.Size([2, 5, 16])
Output:  torch.Size([2, 5, 16])
Forward и backward прошли успешно!


### MiniTransformer

In [25]:
import torch
import torch.nn as nn


class MiniTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_seq_len,
        d_model,
        num_heads,
        num_layers
    ):
        super().__init__()

        # TODO 1: Token + Position Embeddings
        self.embedding = TokenAndPositionEmbedding(vocab_size, max_seq_len, d_model)

        # TODO 2: Последовательность Transformer Blocks
        self.blocks = nn.ModuleList([TransformerBlock(d_model=d_model, num_heads=num_heads) for i in range(num_layers)])

        # TODO 3: Финальная нормализация
        self.ln_f = nn.LayerNorm(d_model)

        # TODO 4: Проекция в размер словаря
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        # token_ids: (B, T)

        # TODO 5: embeddings
        x = self.embedding(token_ids)

        # TODO 6: все Transformer Blocks
        for block in self.blocks:
            x = block(x)
        

        # TODO 7: финальная нормализация
        x = self.ln_f(x)

        # TODO 8: logits
        logits = self.lm_head(x)

        return logits

In [26]:
torch.manual_seed(42)

model = MiniTransformer(
    vocab_size=100,
    max_seq_len=20,
    d_model=32,
    num_heads=4,
    num_layers=2
)

token_ids = torch.randint(0, 100, (2, 10))
logits = model(token_ids)

print("Input:", token_ids.shape)
print("Logits:", logits.shape)

assert logits.shape == (2, 10, 100)
assert torch.isfinite(logits).all()

loss = logits.square().mean()
loss.backward()

assert model.embedding.token_embedding.weight.grad is not None
assert model.blocks[0].attention.query.weight.grad is not None
assert model.blocks[1].ffn[0].weight.grad is not None
assert model.lm_head.weight.grad is not None

print("Forward и backward прошли успешно!")

Input: torch.Size([2, 10])
Logits: torch.Size([2, 10, 100])
Forward и backward прошли успешно!
